In [1]:
import pandas as pd
import os.path
import numpy as np
import datetime as dt

In [2]:
# import Excel sheets # case-sensitive
filename = 'exercise_data.xlsx'
sheet1 = 'data'
sheet2 = 'Viewable Products'
data = pd.read_excel(filename, sheet_name=sheet1)
products = pd.read_excel(filename, sheet_name=sheet2)

In [3]:
data.head(5)

,id,viewable_product_id,created_at,quantity,user_id,removed_at,created_by_client_type,removed_by_client_type
0,1493735,511,2016-07-28 12:34:42.906,1,36867,NaT,www,NaN
1,1607724,490,2016-08-28 19:00:06.840,1,42060,NaT,www,NaN
2,1539095,511,2016-08-10 20:02:59.016,1,52992,2016-12-05 21:31:19.158,www,www
3,1515988,511,2016-08-04 12:47:41.972,1,65654,2016-09-12 11:20:09.225,www,www
4,1495173,511,2016-07-28 19:49:50.478,1,95304,NaT,www,NaN


In [4]:
products.head(5)

,abbrev,viewable_product_id,price,starter_set_count,other_set_count,blade_count,handle_count,shave_gel_count,shave_cream_count,face_wash_count,aftershave_count,lipbalm_count,razorstand_count,face_lotion_count,travel_kit_count
0,ShaveCream,110,8,0,0,0,0,0,1,0,0,0,0,0,0
1,ShaveCream,120,12,0,0,0,0,0,1,0,0,0,0,0,0
2,DailyFaceWash,318,7,0,0,0,0,0,0,1,0,0,0,0,0
3,DailyFaceLotionSPF15,446,8,0,0,0,0,0,0,0,0,0,0,1,0
4,HarrysBlades,488,8,0,0,4,0,0,0,0,0,0,0,0,0


In [5]:
# join two csv files on "viewable_product_id"
df = pd.merge(data, products)
df.head(5)

,id,viewable_product_id,created_at,quantity,user_id,removed_at,created_by_client_type,removed_by_client_type,abbrev,price,...,blade_count,handle_count,shave_gel_count,shave_cream_count,face_wash_count,aftershave_count,lipbalm_count,razorstand_count,face_lotion_count,travel_kit_count
0,1493735,511,2016-07-28 12:34:42.906,1,36867,NaT,www,NaN,BladesPlan,15,...,8,0,0,0,0,0,0,0,0,0
1,1539095,511,2016-08-10 20:02:59.016,1,52992,2016-12-05 21:31:19.158,www,www,BladesPlan,15,...,8,0,0,0,0,0,0,0,0,0
2,1515988,511,2016-08-04 12:47:41.972,1,65654,2016-09-12 11:20:09.225,www,www,BladesPlan,15,...,8,0,0,0,0,0,0,0,0,0
3,1495173,511,2016-07-28 19:49:50.478,1,95304,NaT,www,NaN,BladesPlan,15,...,8,0,0,0,0,0,0,0,0,0
4,1541667,511,2016-08-11 15:33:30.482,1,182104,NaT,www,NaN,BladesPlan,15,...,8,0,0,0,0,0,0,0,0,0


In [6]:
# rename columns
df = df.rename(columns={
    "created_at": "plan_start_date",
    "removed_at": "plan_end_date",
    "created_by_client_type": "plan_start_type",
    "removed_by_client_type": "plan_end_type",
    "abbrev": "product_name",
    "viewable_product_id": "product_id"
})

In [7]:
# replace "www" with "web"
df['plan_start_type'] = df['plan_start_type'].str.replace('www', 'web')
df['plan_end_type'] = df['plan_end_type'].str.replace('www', 'web')

In [8]:
# replace blank values with "on_going" to indicate current subscription
df['plan_end_type'] = df['plan_end_type'].replace(np.nan, "on_going")
df['plan_end_date'] = df['plan_end_date'].replace(np.nan, "on_going")


In [9]:
# Set variable for last date in file
feb012017 = date_time_obj = dt.datetime.strptime('2017-02-01 14:24:00', '%Y-%m-%d %H:%M:%S')

# replace text value 'on_going' with last date in file
df.loc[df.plan_end_date == 'on_going', 'plan_end_date'] = feb012017

In [10]:
# remove rows with data quality issues:
dq_rule_failure = df.loc[
    (df['plan_end_type'] == 'on_going') & df['plan_end_date'].notnull() & (df['plan_end_date'] != feb012017)]

df = df[~df['user_id'].isin(dq_rule_failure['user_id'])]
print("{} rows dropped due to data quality issues".format(dq_rule_failure['user_id'].count()))

77 rows dropped due to data quality issues


In [11]:
#preventative data quality rule 
#dropping rows like the below because a subscription cannot be both on_going and have an 'end_date'
dq_rule_failure.iloc[:5,np.r_[5,7]]

,plan_end_date,plan_end_type
145,2016-11-23 16:30:15.713000,on_going
557,2016-08-02 18:46:07.573000,on_going
588,2016-11-15 22:20:20.320000,on_going
651,2017-01-12 19:07:38.111000,on_going
815,2016-08-15 17:01:23.116000,on_going


In [12]:
# change type of column to datetime
df['plan_end_date'] = pd.to_datetime(df['plan_end_date'], format='%Y-%m-%d %H:%M:%S.%f')

# create columns for the first customer sale date and last customer sale date for the entire customer lifetime
df['first_cust_sale_date'] = (df.groupby(['user_id'])['plan_start_date'].transform('min'))

df['last_cust_sale_date'] = (df.groupby(['user_id'])['plan_start_date'].transform('max'))

In [13]:
df.iloc[:5,np.r_[2,4:9,22:24]]

,plan_start_date,user_id,plan_end_date,plan_start_type,plan_end_type,product_name,first_cust_sale_date,last_cust_sale_date
0,2016-07-28 12:34:42.906,36867,2017-02-01 14:24:00.000,web,on_going,BladesPlan,2016-07-28 12:34:42.906,2016-07-28 12:34:42.906
1,2016-08-10 20:02:59.016,52992,2016-12-05 21:31:19.158,web,web,BladesPlan,2016-08-10 20:02:59.016,2016-08-10 20:02:59.016
2,2016-08-04 12:47:41.972,65654,2016-09-12 11:20:09.225,web,web,BladesPlan,2016-08-04 12:47:41.972,2016-08-04 12:47:41.972
3,2016-07-28 19:49:50.478,95304,2017-02-01 14:24:00.000,web,on_going,BladesPlan,2016-07-28 19:49:50.478,2016-07-28 19:49:50.478
4,2016-08-11 15:33:30.482,182104,2017-02-01 14:24:00.000,web,on_going,BladesPlan,2016-08-11 15:33:30.482,2016-08-11 15:33:30.482


In [14]:
# create temporary column - get the maximum plan_end_date per customer less the first_cust_sale_date column
df['max_plan_end_date_per_user'] = round(
    (df.groupby(['user_id'])['plan_end_date'].transform('max') - df['first_cust_sale_date']).dt.total_seconds() / 86400,
    1)  # where 86400 seconds is one day


# create a function for column creation
def is_on_going(row):
    if row['plan_end_type'] == 'on_going':
        return round(((feb012017 - row['first_cust_sale_date']).total_seconds()) / 86400, 1)
    else:
        return row['max_plan_end_date_per_user']  # temporary column - see above


df['total_cust_lifetime_length_days'] = df.apply(is_on_going, axis=1)  # create a new column based on the function

df['repeat_cust_flag'] = (df['last_cust_sale_date'] - df[
    'first_cust_sale_date']).dt.total_seconds() / 86400 >= 1


In [15]:
df.iloc[:5,np.r_[2,4:9,22:27]]

,plan_start_date,user_id,plan_end_date,plan_start_type,plan_end_type,product_name,first_cust_sale_date,last_cust_sale_date,max_plan_end_date_per_user,total_cust_lifetime_length_days,repeat_cust_flag
0,2016-07-28 12:34:42.906,36867,2017-02-01 14:24:00.000,web,on_going,BladesPlan,2016-07-28 12:34:42.906,2016-07-28 12:34:42.906,188.1,188.1,False
1,2016-08-10 20:02:59.016,52992,2016-12-05 21:31:19.158,web,web,BladesPlan,2016-08-10 20:02:59.016,2016-08-10 20:02:59.016,117.1,117.1,False
2,2016-08-04 12:47:41.972,65654,2016-09-12 11:20:09.225,web,web,BladesPlan,2016-08-04 12:47:41.972,2016-08-04 12:47:41.972,38.9,38.9,False
3,2016-07-28 19:49:50.478,95304,2017-02-01 14:24:00.000,web,on_going,BladesPlan,2016-07-28 19:49:50.478,2016-07-28 19:49:50.478,187.8,187.8,False
4,2016-08-11 15:33:30.482,182104,2017-02-01 14:24:00.000,web,on_going,BladesPlan,2016-08-11 15:33:30.482,2016-08-11 15:33:30.482,174.0,174.0,False


In [16]:
# create a column for line subscription duration days
def insert_name_here(row):
    if row['plan_end_type'] == 'on_going':
        return round(((feb012017 - row['plan_start_date']).total_seconds()) / 86400, 1)
    else:
        return round(((row['plan_end_date'] - row['plan_start_date']).total_seconds()) / 86400, 1)

    
df['line_subscription_duration_days'] = df.apply(insert_name_here, axis=1)  # create a new column based on the functionb

In [17]:
# creation of line_one_time_purchase_flag
df['line_one_time_purchase_flag'] = df['line_subscription_duration_days'] < 0.5

In [18]:
# create a column for subscription_occurrence_count
# if line_subscription_duration_days is less then .5 then "one time purchase"; else count occurrence of said cust_id
def rename_me(row):
    if row['line_subscription_duration_days'] < 0.5:
        return 'one_time_purchase'
    else:
        return df.user_id.eq(
            row.user_id).sum()  # because this code needs to look at the entire df, row[] cannot be used


df['subscription_occurrence_count'] = df.apply(rename_me, axis=1)  # create a new column based on the function


In [19]:
# create cancelled_line_flag
df['cancelled_line_flag'] = np.where(
    df['line_one_time_purchase_flag'],  # condition
    "one_time_purchase",  # value
    df['plan_end_type'] != 'on_going'  # else
)

In [20]:
# create cust_cancellation_count column
# counts the unique number of times a user cancelled a product (aka cancelled_line_flag =='True')
# then uses a merge to apply the distinct count to the df
df = df.merge(
    df.loc[df['cancelled_line_flag'].astype(str) == 'True', 'user_id'].value_counts().rename(
        'cust_cancellation_count'),
    how='outer', left_on='user_id', right_index=True).fillna(
    0)  # uses a full outer join so customers that have not cancelled are not dropped

In [21]:
# create a column for count_of_unique_lifetime_products
# df['count_of_unique_lifetime_products'] = df.groupby('user_id')['product_name'].nunique() # stand alone fucntion before merge
df = df.merge(df.groupby('user_id')['product_name'].nunique().rename('count_of_unique_lifetime_products'),
              how='outer',
              left_on='user_id', right_index=True)


In [22]:
# create a lookup table for a 'Product Type' market lens
product_type_table = pd.DataFrame(
    {'product_name': [
        'ShaveCream', 'DailyFaceWash', 'DailyFaceLotionSPF15', 'HarrysBlades', 'Blades2GelsPlan',
        'Blades,Gel,PostShave', 'FoamingShaveGel',
        'PostShaveBalm', 'BladesPlan', 'RazorBlades', 'Blades1GelPlan', 'ShaveGroomPlan'],

        'product_type': ['Pre/Post Shave', 'FaceCare', 'FaceCare', 'Blades', 'Bundle',
                         'Bundle', 'Pre/Post Shave', 'Pre/Post Shave', 'Blades', 'Blades', 'Bundle', 'Bundle']})

In [23]:
# create a column for product_type
df = df.merge(product_type_table, how='left', on='product_name')

In [24]:
# create a column for the count of sales dates per customer
df['cust_id_occurrence_count'] = df.groupby(['user_id'])['user_id'].transform('count')

In [25]:
# create column for customer_subscription_change_flag
df['customer_subscription_change_flag'] = (df['repeat_cust_flag'] == True) & (df['cust_cancellation_count'] >= 1) & (
        df['cust_id_occurrence_count'] > 1) & (df['count_of_unique_lifetime_products'] > 1)

In [26]:
# create a column for total_cust_life_length_days
def temp(row):
    if ((df["user_id"] == row.user_id) & (df[
                                              "plan_end_type"] == "on_going")).any():  
                                                # where .any() is "if any of the statements are true, this is true".
                                                # Need to understand this better still (can also be witten as sum() > 0
        return round((feb012017 - row['first_cust_sale_date']).total_seconds() / 86400,
                     1)  # where 86400 seconds are in one day
    else:
        return row['max_plan_end_date_per_user']

In [27]:
df['total_cust_life_length_days'] = df.apply(temp, axis=1)  # create a new column based on the function

In [28]:
# set the 'id' column to be the index column
df.set_index("id", inplace=True)

In [29]:
df = df.drop(columns=['max_plan_end_date_per_user'])  # drop temporary column

In [30]:
# reorder columns for export
df = df[[
    'cust_id_occurrence_count',
    'cancelled_line_flag',
    'cust_cancellation_count',
    'line_subscription_duration_days',
    'line_one_time_purchase_flag',
    'first_cust_sale_date',
    'last_cust_sale_date',
    'repeat_cust_flag',
    'count_of_unique_lifetime_products',
    # 'first_subscription_flag', # not including this
    'subscription_occurrence_count',
    'total_cust_lifetime_length_days',
    'customer_subscription_change_flag',
    'product_id',
    'user_id',
    'plan_start_type',
    'plan_end_type',
    'plan_start_date',
    'plan_end_date',
    'product_name',
    'product_type',
    'quantity',
    'price',
    'starter_set_count',
    'other_set_count',
    'blade_count',
    'handle_count',
    'shave_gel_count',
    'shave_cream_count',
    'face_wash_count',
    'aftershave_count',
    'lipbalm_count',
    'razorstand_count',
    'face_lotion_count',
    'travel_kit_count']]

In [31]:
# sort
df = df.sort_values(['user_id', 'id'], ascending=[True, True])

In [32]:
# write to csv
#df.to_csv(os.path.join(r'C:\Users\Jacob Shughrue\Dropbox\Coding\Python\Harrys', 'df_python_export.csv'))